# Compare alternative models — Perpetrator classifier

Este notebook compara el modelo actual de perpetración, basado en red neuronal, contra modelos supervisados alternativos usando el mismo pipeline de datos:

- `lista_global_vars.csv`
- `target_col.csv`
- `scaler_minmax.pkl`
- `medias_escalado.csv`
- `modelo_pca.pkl`
- `splits_indices.json`
- `predictions_with_probs.csv`
- `best_report.json`

La comparación se centra en mantener **recall alto para la clase perpetrador** (`PERPETRADOR = 1`), porque el modelo se usa como herramienta de screening sensible.

> Nota importante: este notebook reconstruye `X_train` y `X_val` desde los ficheros fuente y `splits_indices.json`, en lugar de depender de ficheros llamados `X_train.csv` / `y_train.csv`, para evitar errores por nombres duplicados con el modelo de víctimas.


In [ ]:
# Imports
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
OUTPUT_DIR = Path("compare_models_perpetrator")
OUTPUT_DIR.mkdir(exist_ok=True)

print("OK - entorno preparado")

## 1. Cargar ficheros fuente

Sube al entorno de Colab estos ficheros:

```text
lista_global_vars.csv
target_col.csv
scaler_minmax.pkl
medias_escalado.csv
modelo_pca.pkl
splits_indices.json
predictions_with_probs.csv
best_report.json
```

Opcionalmente también puedes subir `X_val.csv` para comprobar que la reconstrucción coincide con el split original.


In [ ]:
# Load source data
features = pd.read_csv("lista_global_vars.csv")
targets = pd.read_csv("target_col.csv")

print("features:", features.shape)
print("targets:", targets.shape)
print("target columns:", targets.columns.tolist())

## 2. Reconstruir dataset de perpetración

Se aplica el mismo filtro que en el pipeline: eliminar las categorías poco representadas `GENERO_BIN_2` y `ORIENTSEX.BN_3`, y después retirar esas columnas de los predictores.


In [ ]:
# Merge features + perpetrator target
df = pd.concat([features, targets["PERPETRADOR"]], axis=1)

# Same filtering used in the pipeline
df_filtered = df[
    ~((df["GENERO_BIN_2"] == 1) | (df["ORIENTSEX.BN_3"] == 1))
].drop(columns=["GENERO_BIN_2", "ORIENTSEX.BN_3"]).reset_index(drop=True)

X_original = df_filtered.drop(columns=["PERPETRADOR"])
y = df_filtered["PERPETRADOR"].astype(int)

print("Filtered X:", X_original.shape)
print("Filtered y:", y.shape)
print("Target distribution:")
print(y.value_counts().sort_index())

## 3. Aplicar scaler, centrado y PCA

El modelo de perpetración usa los **primeros 22 componentes principales**. El PCA original tiene 27 componentes, pero el clasificador de perpetración usa `PC1`–`PC22`.


In [ ]:
# Load preprocessing artefacts
scaler = joblib.load("scaler_minmax.pkl")
pca = joblib.load("modelo_pca.pkl")
means = pd.read_csv("medias_escalado.csv")["mean"].values

# Apply MinMax scaling, centering, PCA, and keep first 22 PCs
X_scaled = scaler.transform(X_original)
X_centered = X_scaled - means
X_pca_all = pca.transform(X_centered)
X_pca_22 = X_pca_all[:, :22]

print("PCA full:", X_pca_all.shape)
print("PCA perpetrator input:", X_pca_22.shape)

## 4. Reconstruir train/validation usando `splits_indices.json`

Esto evita depender de ficheros `X_train.csv` o `y_train.csv` que pueden tener nombres repetidos entre víctima y perpetrador.


In [ ]:
# Load train/validation split indices
with open("splits_indices.json", "r", encoding="utf-8") as f:
    split_indices = json.load(f)

train_idx = np.array(split_indices["train"])
val_idx = np.array(split_indices["val"])

X_train = X_pca_22[train_idx]
X_val = X_pca_22[val_idx]
y_train = y.iloc[train_idx].values
y_val = y.iloc[val_idx].values

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train distribution:", pd.Series(y_train).value_counts().sort_index().to_dict())
print("y_val distribution:", pd.Series(y_val).value_counts().sort_index().to_dict())

In [ ]:
# Optional consistency check against uploaded X_val.csv if present
xval_path = Path("X_val.csv")
if xval_path.exists():
    X_val_file = pd.read_csv(xval_path).values
    max_abs_diff = np.max(np.abs(X_val_file - X_val))
    print("X_val.csv found")
    print("Max absolute difference between reconstructed X_val and X_val.csv:", max_abs_diff)
else:
    print("X_val.csv not found; skipping consistency check.")

## 5. Métricas auxiliares

Calculamos las métricas de forma homogénea para todos los modelos y diferentes umbrales.


In [ ]:
def evaluate_thresholds(model_name, y_true, y_proba, thresholds):
    rows = []

    auc_pr = average_precision_score(y_true, y_proba)
    try:
        auc_roc = roc_auc_score(y_true, y_proba)
    except ValueError:
        auc_roc = np.nan

    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan

        rows.append({
            "model": model_name,
            "threshold": threshold,
            "recall_1": recall_score(y_true, y_pred, zero_division=0),
            "precision_1": precision_score(y_true, y_pred, zero_division=0),
            "specificity_0": specificity,
            "npv": npv,
            "f1_1": f1_score(y_true, y_pred, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "accuracy": accuracy_score(y_true, y_pred),
            "tp": tp,
            "fp": fp,
            "tn": tn,
            "fn": fn,
            "auc_pr": auc_pr,
            "auc_roc": auc_roc,
        })

    return rows

thresholds = np.round(np.arange(0.15, 0.81, 0.05), 2)
thresholds

## 6. Definir modelos alternativos

Modelos probados:

- Logistic Regression balanceada
- Decision Tree balanceado
- Random Forest balanceado
- Gradient Boosting
- HistGradientBoosting

El modelo de red neuronal actual se añade después usando `predictions_with_probs.csv`.


In [ ]:
models = {
    "LogisticRegression_balanced": LogisticRegression(
        class_weight="balanced",
        max_iter=5000,
        random_state=RANDOM_STATE
    ),
    "DecisionTree_balanced": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
        max_depth=5,
        min_samples_leaf=20
    ),
    "RandomForest_balanced": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        min_samples_leaf=3,
        max_features="sqrt"
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        random_state=RANDOM_STATE,
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=0.1
    ),
}

models

## 7. Entrenar modelos alternativos

Esta celda puede tardar un poco más por Random Forest y los boosting, pero con estos datos debería ejecutarse bien en CPU.


In [ ]:
all_rows = []

for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_val)[:, 1]
    model_rows = evaluate_thresholds(model_name, y_val, y_proba, thresholds)
    all_rows.extend(model_rows)

print("Alternative models evaluated:", len(models))

## 8. Añadir la red neuronal actual

Usamos `predictions_with_probs.csv`, que contiene:

- `y_true`
- `y_pred`
- `y_prob_pos`

Esto permite evaluar la red neuronal actual con distintos umbrales.


In [ ]:
# Load current neural-network predictions
nn_preds = pd.read_csv("predictions_with_probs.csv")

display(nn_preds.head())

y_true_nn = nn_preds["y_true"].astype(int).values
y_proba_nn = nn_preds["y_prob_pos"].astype(float).values

nn_rows = evaluate_thresholds("NeuralNetwork_current", y_true_nn, y_proba_nn, thresholds)
all_rows.extend(nn_rows)

results = pd.DataFrame(all_rows)

print("All results:", results.shape)
display(results.head())

## 9. Comprobar métricas de la red neuronal actual

Con threshold `0.50`, las métricas deberían coincidir con `best_report.json`, salvo pequeñas diferencias de redondeo.


In [ ]:
with open("best_report.json", "r", encoding="utf-8") as f:
    best_report = json.load(f)

print("Best report - class 1:")
print(best_report["1"])

nn_050 = results[
    (results["model"] == "NeuralNetwork_current") &
    (results["threshold"] == 0.50)
]

display(nn_050)

## 10. Comparación con umbral estándar 0.50

Esto muestra el comportamiento de cada modelo si se usa el umbral estándar.


In [ ]:
standard_050 = results[results["threshold"] == 0.50].sort_values(
    by=["balanced_accuracy", "recall_1"],
    ascending=False
)

display(standard_050)

standard_050.to_csv(OUTPUT_DIR / "model_comparison_perpetrator_standard_050.csv", index=False)

## 11. Comparación orientada a recall alto

Aquí filtramos modelos/umbrales con:

```text
recall_1 >= 0.90
```

Este es el criterio más parecido al objetivo del modelo actual.


In [ ]:
high_recall = results[results["recall_1"] >= 0.90].sort_values(
    by=["balanced_accuracy", "precision_1"],
    ascending=False
)

display(high_recall)

high_recall.to_csv(OUTPUT_DIR / "model_comparison_perpetrator_high_recall.csv", index=False)

## 12. Mejor punto por modelo

Seleccionamos, para cada modelo, el mejor punto con `recall_1 >= 0.90`. Si un modelo no alcanza ese recall, no aparece en esta tabla.


In [ ]:
best_high_recall_by_model = (
    high_recall
    .sort_values(by=["model", "balanced_accuracy", "precision_1"], ascending=[True, False, False])
    .groupby("model", as_index=False)
    .head(1)
    .sort_values(by=["balanced_accuracy", "precision_1"], ascending=False)
)

display(best_high_recall_by_model)

best_high_recall_by_model.to_csv(
    OUTPUT_DIR / "model_comparison_perpetrator_best_high_recall_by_model.csv",
    index=False
)

## 13. Guardar todos los resultados

Se generan CSV para añadirlos al repositorio o revisarlos después.


In [ ]:
results.to_csv(OUTPUT_DIR / "model_comparison_perpetrator_results.csv", index=False)

print("Saved files:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", path)

## 14. Interpretación inicial

La lectura recomendada es:

1. Primero mirar `standard_050`: indica qué modelos discriminan mejor con umbral estándar.
2. Después mirar `high_recall`: es la tabla realmente relevante para el objetivo de screening.
3. Comparar la red neuronal actual contra el mejor modelo alternativo con `recall_1 >= 0.90`.
4. Decidir si alguna alternativa mejora de forma clara la red neuronal en:
   - `precision_1`
   - `specificity_0`
   - `balanced_accuracy`
   - `f1_1`
   - `npv`
   - `auc_pr`

Si los modelos alternativos no mejoran claramente a la red neuronal manteniendo recall alto, entonces el resultado refuerza la decisión de mantener la red neuronal para perpetradores.
